# 📬 Webhook Pipeline Verification

This notebook verifies the end-to-end functionality of our FastAPI webhook endpoints (`/api/webhook`), simulating what Meta's Cloud API would send us, without needing a live WhatsApp connection.

In [ ]:
import asyncio
from fastapi.testclient import TestClient
from unittest.mock import AsyncMock

import checkup.messaging.meta_client
from checkup.main import app
from checkup.config import settings

# Create a TestClient for our FastAPI app
client = TestClient(app)

# Mock the meta_client so we don't try to send real WhatsApp messages
checkup.messaging.meta_client.meta_client.send_text = AsyncMock(return_value={"messages": [{"id": "mock_message_id"}]})

print("✅ TestClient and Mocks setup complete.")

### 1. Verification Handshake
When setting up the webhook in Meta Developer Dashboard, Meta sends a GET request to verify we own the endpoint.

In [ ]:
def test_verification_handshake():
    token = settings.meta_verify_token
    challenge = "ch_12345_mock"
    
    # Provide the correct token
    response = client.get(f"/api/webhook?hub.mode=subscribe&hub.verify_token={token}&hub.challenge={challenge}")
    print(f"Correct Token Response: {response.status_code}")
    print(f"Correct Token Body: {response.text}\n")
    
    # Provide an incorrect token
    fail_response = client.get(f"/api/webhook?hub.mode=subscribe&hub.verify_token=wrong_token&hub.challenge={challenge}")
    print(f"Wrong Token Response: {fail_response.status_code}")
    print(f"Wrong Token Body: {fail_response.json()}\n")

test_verification_handshake()

### 2. Inbound Message (Mock Meta Payload)
Now we'll simulate a user sending a message via WhatsApp (a Telugu message). This goes through our FastAPI endpoint, which invokes our LangGraph agent asynchronously, and finally sends a response using the `meta_client` (which we mocked).

In [ ]:
def test_inbound_message():
    # Reset our mock call counter
    checkup.messaging.meta_client.meta_client.send_text.reset_mock()
    
    # Simulated Meta webhook payload for a text message
    mock_payload = {
        "object": "whatsapp_business_account",
        "entry": [
            {
                "id": "WHATSAPP_BUSINESS_ACCOUNT_ID",
                "changes": [
                    {
                        "value": {
                            "messaging_product": "whatsapp",
                            "metadata": {
                                "display_phone_number": "919876543210",
                                "phone_number_id": "PHONE_NUMBER_ID"
                            },
                            "contacts": [
                                {
                                    "profile": {
                                        "name": "Test Parent"
                                    },
                                    "wa_id": "919123456789"
                                }
                            ],
                            "messages": [
                                {
                                    "from": "919123456789",
                                    "id": "wamid.HBgLOTE5...",
                                    "timestamp": "1713456789",
                                    "text": {
                                        "body": "నాకు తలనొప్పి గా ఉంది"  # "I have a headache"
                                    },
                                    "type": "text"
                                }
                            ]
                        },
                        "field": "messages"
                    }
                ]
            }
        ]
    }

    print("🚀 Sending mock inbound message to /api/webhook...")
    
    # FastAPI endpoint processes the message and awaits the graph run
    response = client.post("/api/webhook", json=mock_payload)
    
    print(f"\nFastAPI Response Status: {response.status_code}")
    print(f"FastAPI Response Body: {response.json()}")
    
    # Now let's see what the agent decided to reply with!
    mock = checkup.messaging.meta_client.meta_client.send_text
    if mock.called:
        print("\n✅ The Agent responded!")
        for call in mock.call_args_list:
            args, kwargs = call
            print(f"📤 Sent to {args[0]}:\n{args[1]}")
    else:
        print("\n❌ The agent did not send a response through meta_client.")

test_inbound_message()

✅ If you see the agent's response in Telugu above, the full webhook pipeline works perfectly!